# Lab 5b: Capped AWS Educate/Academy Credit Render Task & OpenUSD Cross-Pillar Audit
**Course:** CIS 536 / 736 - Computer Graphics (Fall 2026)  
**Phase 4:** Integration & Application | **Module 9:** Cloud Render Farms — AWS Deadline Cloud (Lectures 28–30)  
**Associated Assignments:** PS 8 (Term Project Interim Report) & MP 9 (Cloud Render Farm Task)

---

## Objective & Capped Credit Policy
In **Module 9 (Lectures 28–30)**, you transition from single-workstation execution to distributed cloud rendering on **AWS Deadline Cloud** while preparing your **PS 8 Interim Report** cross-pillar OpenUSD checkpoint:

> **🚨 Capped Credit Allocation Warning (MP 9 Policy):** You have a strictly capped AWS Educate/Academy credit allocation for this task. Exhausting your allocation due to an unbudgeted high-resolution or high-sample render is a natural consequence and part of the learning objective—no supplemental credits will be issued. **Never submit a cloud job before running Part 1 of this notebook.**

### Notebook Workflow
1. **Part 1 (Pre-Submission Compute-Hour & Credit Budget Calculator — MP 9 Part 1):** Calculate your exact compute-hour and dollar/credit estimate from frame count, resolution, path-tracing sample count, fleet instance rate, and worker startup overhead, and evaluate a concrete scene-complexity reduction (LOD / sample reduction).
2. **Part 2 (OpenUSD Cross-Pillar Composition & Sublayer Audit — PS 8 Parts 3 & 4):** Inspect a multi-pillar OpenUSD stage to verify $\ge 2$ course pillars are composed via OpenUSD arcs (`subLayers`, `references`, `payloads`, `variantSets`), check for broken absolute paths before cloud upload, and auto-generate the sublayer hierarchy diagram required for **PS 8 Part 4**.
3. **Part 3 (OpenJD / AWS Deadline Cloud Job Bundle & Budget Gate — Lectures 29 & 30):** Generate a deterministic Open Job Description (`template.yaml`) and parameter bundle for AWS Deadline Cloud.
4. **Part 4 (Local-vs-Cloud Render Parity & Cost Discrepancy Analysis — MP 9 Parts 2 & 3):** Verify output frame lighting/material consistency (PSNR / SSIM) against a local reference frame and compute your formal **Estimated vs. Actual Cost Discrepancy Analysis**.

## Part 1: Pre-Submission Compute-Hour & Capped Credit Budget Calculator (MP 9 Part 1)
Before submitting a job to an AWS Deadline Cloud queue, we model total credit consumption $C_{\text{est}}$ as a function of:
* **Frame Count ($N_{\text{frames}}$):** Total animation frames submitted to the queue.
* **Resolution Scaling Factor ($s_{\text{res}}$):** Relative to your local benchmark frame $(W_0, H_0)$, $s_{\text{res}} = \frac{W \cdot H}{W_0 \cdot H_0}$.
* **Sample Count Scaling Factor ($s_{\text{spp}}$):** Linear scaling of Cycles/path-tracing samples per pixel $S_{\text{spp}} / S_0$.
* **Fleet Provisioning & I/O Overhead ($t_{\text{overhead}}$):** Per-worker container startup, OpenUSD/texture sync from S3 Job Attachments, and scene BVH build time.

Execute the cell below to compare a naive configuration against a cost-optimized configuration (with an explicit scene-complexity reduction) against your capped credit ceiling.

In [ ]:
from dataclasses import dataclass
import math
import json
from pathlib import Path
import numpy as np

@dataclass
class RenderJobSpec:
    name: str
    frame_count: int
    width: int
    height: int
    samples_per_pixel: int
    fleet_workers: int
    instance_type: str
    hourly_rate_usd: float
    # Local 1-frame benchmark anchor (e.g., 1080p at 128 samples took 45 sec on reference GPU)
    ref_frame_sec: float = 45.0
    ref_width: int = 1920
    ref_height: int = 1080
    ref_samples: int = 128
    worker_startup_sec: float = 90.0
    task_chunk_size: int = 5
    scene_io_sec_per_task: float = 12.0

def estimate_deadline_cloud_cost(spec: RenderJobSpec, credit_cap_usd: float = 10.0) -> dict:
    res_scale = (spec.width * spec.height) / (spec.ref_width * spec.ref_height)
    spp_scale = spec.samples_per_pixel / spec.ref_samples
    est_sec_per_frame = spec.ref_frame_sec * res_scale * spp_scale

    num_tasks = math.ceil(spec.frame_count / spec.task_chunk_size)
    active_workers = min(spec.fleet_workers, num_tasks)

    total_render_sec = spec.frame_count * est_sec_per_frame
    total_io_sec = num_tasks * spec.scene_io_sec_per_task
    total_startup_sec = active_workers * spec.worker_startup_sec

    total_compute_sec = total_render_sec + total_io_sec + total_startup_sec
    compute_hours = total_compute_sec / 3600.0
    wall_clock_min = (total_compute_sec / active_workers) / 60.0
    estimated_cost_usd = compute_hours * spec.hourly_rate_usd
    overhead_share_pct = 100.0 * (total_io_sec + total_startup_sec) / total_compute_sec
    within_budget = estimated_cost_usd <= (0.80 * credit_cap_usd) # 20% safety margin

    return {
        "name": spec.name,
        "instance_type": spec.instance_type,
        "frames": spec.frame_count,
        "resolution": f"{spec.width}x{spec.height}",
        "samples": spec.samples_per_pixel,
        "workers": active_workers,
        "sec_per_frame": round(est_sec_per_frame, 2),
        "compute_hours": round(compute_hours, 3),
        "wall_clock_min": round(wall_clock_min, 2),
        "overhead_pct": round(overhead_share_pct, 1),
        "estimated_cost_usd": round(estimated_cost_usd, 2),
        "credit_cap_usd": credit_cap_usd,
        "status": "APPROVED (<=80% Cap)" if within_budget else "REJECTED (Exceeds 80% Safety Cap)"
    }

CREDIT_CAP = 10.00  # Example capped allocation ($10.00 credit ceiling for MP 9)

# Plan A: Unoptimized 4K, 512 SPP, chunk_size=1 on 8 on-demand g5.2xlarge workers
unoptimized_job = RenderJobSpec(
    name="Plan A (Unoptimized 4K / 512 SPP)",
    frame_count=120, width=3840, height=2160, samples_per_pixel=512,
    fleet_workers=8, instance_type="g5.2xlarge (On-Demand)", hourly_rate_usd=1.212,
    task_chunk_size=1
)

# Plan B: Cost-Optimized 1080p, 128 SPP + OptiX Denoiser + LOD Variant, chunk_size=10 on 4 Spot g5.xlarge workers
optimized_job = RenderJobSpec(
    name="Plan B (1080p / 128 SPP + Denoiser + LOD)",
    frame_count=120, width=1920, height=1080, samples_per_pixel=128,
    fleet_workers=4, instance_type="g5.xlarge (Spot)", hourly_rate_usd=0.402,
    ref_frame_sec=32.0, # 29% faster per frame after OpenUSD LOD complexity reduction
    task_chunk_size=10
)

print("=== MP 9 Part 1: Pre-Submission AWS Deadline Cloud Budget Comparison ===")
for job in [unoptimized_job, optimized_job]:
    res = estimate_deadline_cloud_cost(job, credit_cap_usd=CREDIT_CAP)
    print(
        f"{res['name']:42s} | {res['resolution']} @ {res['samples']} SPP | "
        f"Compute: {res['compute_hours']:6.2f} hrs | Wall: {res['wall_clock_min']:5.1f}m | "
        f"Overhead: {res['overhead_pct']:4.1f}% | Est Cost: ${res['estimated_cost_usd']:5.2f} / ${res['credit_cap_usd']:.2f} -> {res['status']}"
    )

## Part 2: OpenUSD Cross-Pillar Composition & Sublayer Audit (PS 8 Parts 3 & 4)
For **PS 8 (Term Project Interim Report)**, you must demonstrate that your project integrates **at least 2 course pillars** (e.g., *Pillar 1: Procedural Geometry Nodes*, *Pillar 2: Rigging & Animation*, *Pillar 3: Generative Geometry / 3DGS*, *Pillar 4: Cloud Rendering*, *Pillar 5: Data Visualization*) inside a unified **OpenUSD stage**, and provide a diagram/hierarchy of your sublayers and composition arcs (**LIVRPS**: Local, Inherits, VariantSets, References, Payloads, Specializes).

Furthermore, before submitting a scene to AWS Deadline Cloud (**MP 9**), you must verify that all asset references use **relative paths** (absolute workstation paths like `C:\Users\...` will fail on Linux cloud worker nodes).

In [ ]:
import re

LAB5B_DIR = Path.cwd() / "lab5b_workspace"
LAB5B_DIR.mkdir(parents=True, exist_ok=True)

# Create a sample multi-pillar root OpenUSD stage (.usda) demonstrating PS 8 composition arcs
sample_root_usda = '''#usda 1.0
(
    defaultPrim = "World"
    metersPerUnit = 1.0
    upAxis = "Y"
    subLayers = [
        @./layers/lighting_hdrp_override.usda@,
        @./layers/procedural_environment_base.usda@
    ]
)

def Xform "World"
{
    def Xform "ProceduralTerrain" (
        prepend references = @./assets/geometry_nodes_canyon.usda@
        variants = {
            string lodVariant = "cloud_medium"
        }
        prepend variantSets = "lodVariant"
    )
    {
        variantSet "lodVariant" = {
            "high_local" {
                int polyBudget = 500000
            }
            "cloud_medium" {
                int polyBudget = 125000
            }
        }
    }

    def Xform "GenerativeLanternProp" (
        prepend payload = @./assets/mp7_generative_asset.usda@
    )
    {
        double3 xformOp:translate = (2.5, 0.0, -1.2)
        uniform token[] xformOpOrder = ["xformOp:translate"]
    }

    def Xform "RiggedExplorerCharacter" (
        prepend references = @./assets/rigged_explorer_anim.usda@
    )
    {
    }
}
'''

root_stage_path = LAB5B_DIR / "ps8_cross_pillar_stage.usda"
root_stage_path.write_text(sample_root_usda, encoding="utf-8")

def audit_openusd_stage_for_ps8_and_cloud(usda_path: Path):
    text = usda_path.read_text(encoding="utf-8")
    asset_paths = re.findall(r"@([^@]+)@", text)
    variant_sets = re.findall(r'variantSet\s+"([^"]+)"', text)
    references = re.findall(r"references\s*=\s*@([^@]+)@", text)
    payloads = re.findall(r"payload\s*=\s*@([^@]+)@", text)

    bad_abs_paths = [p for p in asset_paths if (":" in p or p.startswith("/") or p.startswith("\\"))]

    print(f"=== PS 8 & MP 9 OpenUSD Stage Audit: {usda_path.name} ===")
    print("\n[PS 8 Part 4 — Auto-Generated Sublayer & Composition-Arc Hierarchy]")
    print(f"└── Root Stage: {usda_path.name} (defaultPrim = World, upAxis = Y, metersPerUnit = 1.0)")
    for p in asset_paths:
        if p not in references and p not in payloads:
            print(f"    ├── [SubLayer Arc]   @{p}@")
    for r in references:
        print(f"    ├── [Reference Arc]  @{r}@")
    for pl in payloads:
        print(f"    ├── [Payload Arc]    @{pl}@")
    for vs in sorted(set(variant_sets)):
        print(f"    └── [VariantSet Arc] '{vs}' (Used for MP 9 Cloud LOD Complexity Reduction)")

    print("\n[MP 9 Cloud Job Attachments Path Hygiene Check]")
    if bad_abs_paths:
        print(f"  FAILED: Found non-portable absolute paths: {bad_abs_paths}")
    else:
        print(f"  PASSED: All {len(asset_paths)} @asset@ paths are relative and portable to AWS Deadline Cloud workers.")

audit_openusd_stage_for_ps8_and_cloud(root_stage_path)

## Part 3: Authoring the AWS Deadline Cloud OpenJD Template & Job Bundle (Lectures 29 & 30)
AWS Deadline Cloud uses the **Open Job Description (OpenJD)** specification (`specificationVersion: 'jobtemplate-2023-09'`) to parameterize distributed render jobs across worker fleets.

The cell below generates a complete `job_bundle/` directory containing:
1. `template.yaml` — The OpenJD job template defining frame range, resolution, sample count, and the headless Blender 4.2 / OpenUSD command.
2. `parameter_values.yaml` — The locked, budget-verified parameters from **Part 1 (Plan B)** so you never accidentally launch an unbudgeted 4K render.

In [ ]:
BUNDLE_DIR = LAB5B_DIR / "mp9_deadline_job_bundle"
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

openjd_template_yaml = '''specificationVersion: 'jobtemplate-2023-09'
name: CIS536_736_MP9_Cloud_Render
description: Budget-gated Blender 4.2 / OpenUSD distributed render job for CIS 536/736 MP 9.
parameterDefinitions:
  - name: SceneFile
    type: PATH
    objectType: FILE
    dataFlow: IN
    description: Input Blender or OpenUSD scene file.
  - name: OutputDir
    type: PATH
    objectType: DIRECTORY
    dataFlow: OUT
    description: Output directory for rendered PNG frames.
  - name: Frames
    type: STRING
    default: '1-120'
  - name: ResolutionX
    type: INT
    default: 1920
  - name: ResolutionY
    type: INT
    default: 1080
  - name: Samples
    type: INT
    default: 128
steps:
  - name: RenderFrames
    parameterSpace:
      taskParameterDefinitions:
        - name: Frame
          type: INT
          range: '{{Param.Frames}}'
    script:
      actions:
        onRun:
          command: blender
          args:
            - '-b'
            - '{{Param.SceneFile}}'
            - '-E'
            - 'CYCLES'
            - '-o'
            - '{{Param.OutputDir}}/frame_####'
            - '-F'
            - 'PNG'
            - '-f'
            - '{{Task.Param.Frame}}'
            - '--'
            - '--cycles-device'
            - 'CUDA'
'''

parameter_values_yaml = '''parameterValues:
  - name: SceneFile
    value: ./ps8_cross_pillar_stage.blend
  - name: OutputDir
    value: ./renders
  - name: Frames
    value: '1-120'
  - name: ResolutionX
    value: 1920
  - name: ResolutionY
    value: 1080
  - name: Samples
    value: 128
'''

(BUNDLE_DIR / "template.yaml").write_text(openjd_template_yaml, encoding="utf-8")
(BUNDLE_DIR / "parameter_values.yaml").write_text(parameter_values_yaml, encoding="utf-8")

print(f"[OpenJD] Wrote job bundle to: {BUNDLE_DIR}")
print("\n=== AWS Deadline Cloud CLI Submission Commands ===")
print(f"1. Validate bundle locally : openjd check {BUNDLE_DIR / 'template.yaml'}")
print(f"2. Estimate/Submit to queue: deadline bundle submit {BUNDLE_DIR} --parameter-file {BUNDLE_DIR / 'parameter_values.yaml'} --yes")

## Part 4: Post-Render Output Verification & Cost Discrepancy Analysis (MP 9 Parts 2 & 3)
Once your AWS Deadline Cloud render job completes, **MP 9 Parts 2 and 3** require you to:
1. **Verify Output Correctness (25 pts):** Confirm that the cloud-rendered output matches your local reference render in lighting, PBR materials, and color space (e.g., no missing texture pink shaders or Gamma/AgX mismatches).
2. **Analyze Estimated vs. Actual Cost Discrepancy (20 pts):** Compare your pre-submission compute-hour/credit estimate from Part 1 against the actual telemetry reported by AWS Deadline Cloud, and explain the sources of variance (cold-start fleet provisioning, S3 Job Attachment sync latency, and per-frame ray-bounce variance as camera angles change).

In [ ]:
def analyze_render_parity_and_cost_discrepancy(
    local_ref_img: np.ndarray,
    cloud_render_img: np.ndarray,
    estimated_summary: dict,
    actual_compute_hours: float,
    actual_cost_usd: float,
    actual_wall_clock_min: float
):
    # 1. Local vs. Cloud Render Lighting/Material Parity Check (MSE & PSNR)
    mse = float(np.mean((local_ref_img.astype(np.float64) - cloud_render_img.astype(np.float64)) ** 2))
    psnr = float("inf") if mse == 0 else 20.0 * math.log10(255.0 / math.sqrt(mse))
    parity_pass = psnr >= 30.0

    # 2. Cost Discrepancy Calculation
    est_hrs = estimated_summary["compute_hours"]
    est_cost = estimated_summary["estimated_cost_usd"]
    delta_hrs = actual_compute_hours - est_hrs
    delta_cost = actual_cost_usd - est_cost
    pct_err_cost = 100.0 * delta_cost / max(est_cost, 1e-6)

    print("=== MP 9 Part 2: Local vs. Cloud Render Quality Verification ===")
    print(f"Frame MSE : {mse:.4f} | PSNR: {psnr:.2f} dB -> {'PASSED (Matches Local Lighting/Materials)' if parity_pass else 'WARNING: Check Missing Textures or Color Management'}")

    print("\n=== MP 9 Part 3: Pre-Submission Estimate vs. Actual Cloud Cost Analysis ===")
    print(f"Metric                | Pre-Submission Estimate | Actual Cloud Telemetry | Discrepancy (Delta)")
    print(f"----------------------|-------------------------|------------------------|--------------------")
    print(f"Compute Hours (hrs)   | {est_hrs:23.3f} | {actual_compute_hours:22.3f} | {delta_hrs:+18.3f} hrs")
    print(f"Wall-Clock Time (min) | {estimated_summary['wall_clock_min']:23.2f} | {actual_wall_clock_min:22.2f} | {actual_wall_clock_min - estimated_summary['wall_clock_min']:+18.2f} min")
    print(f"Total Cost (USD)      | ${est_cost:22.2f} | ${actual_cost_usd:21.2f} | ${delta_cost:+17.2f} ({pct_err_cost:+.1f}%)")
    print(f"Remaining Credit Pool | ${estimated_summary['credit_cap_usd'] - est_cost:22.2f} | ${estimated_summary['credit_cap_usd'] - actual_cost_usd:21.2f} | Cap = ${estimated_summary['credit_cap_usd']:.2f}")

# Simulate verification on a synthetic 1080p reference frame (with minor Monte Carlo path-tracing noise)
rng = np.random.default_rng(536)
local_frame = rng.integers(20, 235, size=(270, 480, 3), dtype=np.uint8)
cloud_frame = np.clip(local_frame.astype(np.int16) + rng.integers(-2, 3, size=local_frame.shape), 0, 255).astype(np.uint8)

plan_b_est = estimate_deadline_cloud_cost(optimized_job, credit_cap_usd=CREDIT_CAP)
analyze_render_parity_and_cost_discrepancy(
    local_ref_img=local_frame,
    cloud_render_img=cloud_frame,
    estimated_summary=plan_b_est,
    actual_compute_hours=1.340,
    actual_cost_usd=0.54,
    actual_wall_clock_min=20.10
)

## Part 5: PS 8 & MP 9 Submission Checklist
1. **For PS 8 (Term Project Interim Report — Due Thu 12 Nov 2026, Mandatory Undroppable Milestone):**
   * 400–600 word Progress Narrative confirming your $\ge 2$ integrated pillars.
   * Single biggest remaining technical risk and concrete mitigation plan.
   * Screenshot/capture showing assets from both pillars composed together in a single OpenUSD-driven Unity 6 scene.
   * OpenUSD Sublayer & Composition-Arc hierarchy diagram (generated in **Part 2** above).
   * Week-by-week remaining schedule through HW 12 + `GenAI.pdf` audit log.
2. **For MP 9 (Cloud Render Farm Task — Due Sat 21 Nov 2026):**
   * Pre-submission configuration & budget table + scene-complexity reduction strategy (**Part 1** above).
   * OpenJD `template.yaml` / `parameter_values.yaml` committed to GitHub Classroom (**Part 3** above) + rendered output frame(s) matching local lighting/materials.
   * Estimated vs. Actual compute-hour and credit discrepancy analysis (**Part 4** above) + `GenAI.pdf` audit log.